In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set the path to your data folder
data_path = Path(r'C:\Users\WELCOME\Downloads\blinkit_dataset\mini-project')

# Check if we can see the files (Sanity Check)
files = list(data_path.glob("*.csv")) + list(data_path.glob("*.xlsx"))
print(f"Found {len(files)} files.")

Found 11 files.


In [5]:
# Create a dictionary to hold all dataframes
dfs = {}

# Load CSVs
for file in data_path.glob("*.csv"):
    name = file.stem # Gets the filename without .csv
    dfs[name] = pd.read_csv(file)

# Load Excel files (using openpyxl you installed earlier)
for file in data_path.glob("*.xlsx"):
    name = file.stem
    dfs[name] = pd.read_excel(file)

print("All files loaded into the 'dfs' dictionary!")

All files loaded into the 'dfs' dictionary!


In [6]:
# 1. Start with the core sales: Orders + Order Items
master_df = pd.merge(dfs['blinkit_orders'], dfs['blinkit_order_items'], on='order_id', how='left')

# 2. Add Product Details (Category, Brand, Price)
master_df = pd.merge(master_df, dfs['blinkit_products'], on='product_id', how='left')

# 3. Add Delivery Performance (Speed and Status)
master_df = pd.merge(master_df, dfs['blinkit_delivery_performance'], on='order_id', how='left')

# 4. Add Customer Profile (Area and Segment)
master_df = pd.merge(master_df, dfs['blinkit_customers'], on='customer_id', how='left')

# 5. Add Feedback (Sentiment and Ratings)
master_df = pd.merge(master_df, dfs['blinkit_customer_feedback'], on=['order_id', 'customer_id'], how='left')

print(f"Master Merge Complete! Shape: {master_df.shape}")

Master Merge Complete! Shape: (5000, 45)


In [7]:
# Convert dates to datetime objects
date_cols = ['order_date', 'actual_time', 'promised_time']
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col])

# Calculate Delivery Delay in Minutes
master_df['delay_minutes'] = (master_df['actual_time'] - master_df['promised_time']).dt.total_seconds() / 60

# Fill missing feedback ratings with 0 or 'None'
master_df['rating'] = master_df['rating'].fillna(0)
master_df['feedback_text'] = master_df['feedback_text'].fillna("No Feedback")

# Save your clean "Golden Dataset" for the dashboard
master_df.to_csv('Blinkit_Master_Data.csv', index=False)
print("Golden Dataset saved as 'Blinkit_Master_Data.csv'")

Golden Dataset saved as 'Blinkit_Master_Data.csv'
